<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-07-multi-agent-content-pipeline-for-orbit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7 (graded) — Multi-agent content pipeline for Orbit — is it worth it?
**Course 3: AI Agents and Agentic AI with Python — Chapter 7: Multi-agent systems**

**Problem brief (Leo Farkas, Orbit Retail):** "Marketing wants a 'research → draft →
fact-check → edit' content pipeline. One mega-prompt agent does all four badly. Would
specialist agents actually help — and what does that cost?"

**What you'll submit:** a supervisor + specialist pipeline, run against a single-agent
baseline and a fixed workflow, with a real quality/cost/latency comparison and a cost ledger.
A well-supported "use the workflow" conclusion earns full marks.

## 1. The task and a quality checklist
This lab uses deterministic role functions (not a live LLM) so the three architectures are
compared on a level, reproducible playing field — the *pattern* generalizes directly to real
LLM calls; swap `call_llm_as(role, ...)` for a real API call to run this for real.

In [ ]:
task = 'Write a short product blurb for the Orbit ComfortFit wireless earbuds, using real review themes.'

REVIEWS = [
    'Battery lasts a full day, very comfortable for long wear.',
    'Bass is punchy but mids are a bit muddy.',
    'Bluetooth connection dropped a few times in the first week.',
]

CHECKLIST = ['mentions battery life', 'mentions comfort', 'mentions a caveat/limitation', 'has a call to action']

def score_against_checklist(text):
    t = text.lower()
    hits = {
        'mentions battery life': 'battery' in t,
        'mentions comfort': 'comfort' in t,
        'mentions a caveat/limitation': any(w in t for w in ['however', 'occasional', 'some users', 'mixed', 'caveat']),
        'has a call to action': any(w in t for w in ['shop', 'buy', 'order', 'get yours', 'try']),
    }
    return sum(hits.values()) / len(CHECKLIST), hits

## 2. Architecture 1: a single mega-prompt agent

In [ ]:
def single_agent_pipeline():
    """One overloaded call trying to research, write, fact-check, AND edit at once — the
    'does all four badly' failure mode Leo described. Simulated as a deterministic function
    that (realistically) drops the caveat and the CTA under the combined cognitive load."""
    calls = 1
    output = ('The Orbit ComfortFit wireless earbuds deliver all-day battery life and a '
               'comfortable fit for extended wear, with punchy bass for your favorite tracks.')
    return output, calls

## 3. Architecture 2: a fixed 4-step workflow

In [ ]:
def fixed_workflow_pipeline():
    """Same four steps, always run in the same order, no agent deciding anything — just
    function calls chained together. Each step focuses on ONE job."""
    calls = 0

    calls += 1
    research_notes = '; '.join(REVIEWS)

    calls += 1
    draft = (f'The Orbit ComfortFit earbuds get praise for all-day battery life and a '
              f'comfortable fit. Bass is punchy, though some users note the mids and an '
              f'occasional Bluetooth hiccup. Shop ComfortFit today.')

    calls += 1  # fact-check: verify every claim in the draft traces back to a real review
    claim_keywords = ['battery', 'comfort', 'bass', 'bluetooth']
    claims_ok = all(kw in research_notes.lower() for kw in claim_keywords if kw in draft.lower())

    calls += 1  # edit pass: tighten wording
    final = draft.replace('  ', ' ')

    return final, calls

## 4. Architecture 3: a supervisor + specialist agents (with a critique loop)

In [ ]:
def multi_agent_pipeline(max_revisions=1):
    """A supervisor delegates to specialist roles, each with a narrow, focused context. The
    fact-checker can send the draft back to the writer once — this adversarial check is the
    thing a single fixed workflow doesn't naturally have."""
    calls = 0

    calls += 1  # supervisor delegates to researcher
    research_notes = '; '.join(REVIEWS)

    calls += 1  # writer
    draft = ('The Orbit ComfortFit earbuds deliver all-day battery life and all-day comfort, '
              'with punchy bass. Shop now.')

    revisions = 0
    while revisions <= max_revisions:
        calls += 1  # fact-checker / critic — specifically checks for a missing caveat
        has_caveat = any(w in draft.lower() for w in ['however', 'occasional', 'some users', 'mixed'])
        if has_caveat:
            break
        calls += 1  # writer revises based on the critique
        draft = draft.replace('with punchy bass.', 'with punchy bass, though some users note occasional Bluetooth drops.')
        revisions += 1

    calls += 1  # editor
    final = draft
    return final, calls

## 5. Run the three-way comparison

In [ ]:
import pandas as pd

COST_PER_CALL_USD = 0.0003  # a placeholder per-call cost; use your provider's real rate

results = []
for name, fn in [('single-agent', single_agent_pipeline), ('fixed-workflow', fixed_workflow_pipeline),
                  ('multi-agent', multi_agent_pipeline)]:
    output, calls = fn()
    quality, hits = score_against_checklist(output)
    results.append({'architecture': name, 'quality_score': quality, 'llm_calls': calls,
                     'est_cost_usd': round(calls * COST_PER_CALL_USD, 5), 'output': output})

comparison_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 100)
comparison_df[['architecture', 'quality_score', 'llm_calls', 'est_cost_usd']]

In [ ]:
for r in results:
    print(f"=== {r['architecture']} ===\n{r['output']}\n")

## 6. Cost ledger at Orbit's real volume

In [ ]:
MONTHLY_BLURBS = 2000
ledger = comparison_df[['architecture', 'llm_calls', 'est_cost_usd']].copy()
ledger['est_monthly_cost_usd'] = (ledger['est_cost_usd'] * MONTHLY_BLURBS).round(2)
ledger

## 7. Recommendation (fill in)
Using the quality scores and cost ledger above: does the multi-agent pipeline's quality
improvement over the fixed workflow justify its extra calls at Orbit's real volume? Would
your answer change at 10x the volume, or for a task where quality matters more than cost
(e.g. a legal disclosure, not a product blurb)? A well-supported "use the workflow"
conclusion is a fully correct answer here.

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 7: Multi-agent systems*